# SASRec Stage 3 Refine Sinusoidal Multi-Task BPI2012 Colab Train 10

This notebook tests Stage 3 multi-task learning with the
`refine_ml50_do035 + delta_start + sinusoidal` time-aware setting.

New experiments:
- `refine_sinusoidal_multi_task_w1.0`
- `refine_sinusoidal_multi_task_w0.1`

Main comparison groups:
- `refine_baseline`
- `refine_attnbias_single_task`
- `refine_sinusoidal_single_task`
- `refine_multi_task_w1.0`
- `refine_attnbias_multi_task_w1.0`
- `refine_sinusoidal_multi_task_w1.0`
- `refine_sinusoidal_multi_task_w0.1`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_bpi2012_ndcg10'
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_timeaware_attention_bias_ndcg10'
SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_timeaware_sinusoidal_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2'
MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_stage3_refine_sinusoidal_multitask_ndcg10_v1'
MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_stage3_refine_sinusoidal_multitask_w01_ndcg10_v1'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR:', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
print('SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR:', SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR:', MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR)
print('MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR:', MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR)
print('MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR:', MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2
MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refin

In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR"
!mkdir -p "$SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [8]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260614_055854
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = ['delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

This notebook tests the `refine_ml50_do035 + sinusoidal + delta_start_seconds` setting with:
- multi-task `time_loss_weight=1.0`
- multi-task `time_loss_weight=0.1`

Fixed settings:
- backbone: `refine_ml50_do035`
- time-aware input: `delta_start_seconds`
- time encoding: `sinusoidal`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`


## Check prerequisite reference runs


In [11]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
single_task_sinusoidal_runs = [
    'timeaware_dstart_sinusoidal_ml50_do035_s42',
    'timeaware_dstart_sinusoidal_ml50_do035_s2024',
    'timeaware_dstart_sinusoidal_ml50_do035_s7',
]
multitask_baseline_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]
multitask_attnbias_runs = [
    'multitask_attnbias_dstart_ml50_do035_b9_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_s7',
]

checks = [
    ('baseline', BASELINE_NDCG10_OUTPUT_DIR, baseline_runs),
    ('single-task attention bias', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR, single_task_attnbias_runs),
    ('single-task sinusoidal', SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR, single_task_sinusoidal_runs),
    ('multi-task baseline', MULTITASK_BASELINE_OUTPUT_DIR, multitask_baseline_runs),
    ('multi-task attention bias', MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR, multitask_attnbias_runs),
]

print('=' * 80)
for label, output_dir, run_names in checks:
    print(label)
    base = Path(output_dir)
    for run_name in run_names:
        run_dir = base / run_name
        print(' ', run_name, 'EXISTS' if run_dir.exists() else 'MISSING')
    print('-' * 80)


baseline
  refine_ml50_do035_s42 EXISTS
  refine_ml50_do035_s2024 EXISTS
  refine_ml50_do035_s7 EXISTS
--------------------------------------------------------------------------------
single-task attention bias
  attnbias_dstart_ml50_do035_b9_s42 EXISTS
  attnbias_dstart_ml50_do035_b9_s2024 EXISTS
  attnbias_dstart_ml50_do035_b9_s7 EXISTS
--------------------------------------------------------------------------------
single-task sinusoidal
  timeaware_dstart_sinusoidal_ml50_do035_s42 EXISTS
  timeaware_dstart_sinusoidal_ml50_do035_s2024 EXISTS
  timeaware_dstart_sinusoidal_ml50_do035_s7 EXISTS
--------------------------------------------------------------------------------
multi-task baseline
  multitask_refine_ml50_do035_s42 EXISTS
  multitask_refine_ml50_do035_s2024 EXISTS
  multitask_refine_ml50_do035_s7 EXISTS
--------------------------------------------------------------------------------
multi-task attention bias
  multitask_attnbias_dstart_ml50_do035_b9_s42 EXISTS
  multitask_a

## Check planned refine sinusoidal multi-task runs


In [12]:
planned_sinusoidal_multitask_runs = [
    'multitask_sinusoidal_dstart_ml50_do035_s42',
    'multitask_sinusoidal_dstart_ml50_do035_s2024',
    'multitask_sinusoidal_dstart_ml50_do035_s7',
    'multitask_sinusoidal_dstart_ml50_do035_w01_s42',
    'multitask_sinusoidal_dstart_ml50_do035_w01_s2024',
    'multitask_sinusoidal_dstart_ml50_do035_w01_s7',
]

checks = [
    ('w1.0', MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR),
    ('w0.1', MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR),
]

for label, output_dir in checks:
    base = Path(output_dir)
    print('=' * 80)
    print(f'Stage 3 refine sinusoidal multi-task runs ({label})')
    for run_name in planned_sinusoidal_multitask_runs:
        if (label == 'w1.0' and '_w01_' in run_name) or (label == 'w0.1' and '_w01_' not in run_name):
            continue
        run_dir = base / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 refine sinusoidal multi-task runs (w1.0)
multitask_sinusoidal_dstart_ml50_do035_s42 OK
multitask_sinusoidal_dstart_ml50_do035_s2024 OK
multitask_sinusoidal_dstart_ml50_do035_s7 OK
Stage 3 refine sinusoidal multi-task runs (w0.1)
multitask_sinusoidal_dstart_ml50_do035_w01_s42 OK
multitask_sinusoidal_dstart_ml50_do035_w01_s2024 OK
multitask_sinusoidal_dstart_ml50_do035_w01_s7 OK


## Train refine sinusoidal multi-task w1.0 runs


### `multitask_sinusoidal_dstart_ml50_do035_s42`


In [13]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_s42
epoch=1, loss=3.0630
epoch=2, loss=1.7418
epoch=3, loss=1.5198
epoch=4, loss=1.4067
epoch=5, loss=1.3391
valid [task], Top5Acc: 0.4277, Top10Acc: 0.7182, Acc: 0.0517, MacroF1: 0.0710, TimeMAE: 69017.3724, TimeRMSE: 276935.1032, TimeMedAE: 1538.7991
valid [full], NDCG@5: 0.5866, HR@5: 0.6815, NDCG@10: 0.6812, HR@10: 0.9735, MRR: 0.5978
valid [sampled], NDCG@5: 0.5042, HR@5: 0.5063, NDCG@10: 0.5092, HR@10: 0.5219, MRR: 0.5197
test [task], Top5Acc: 0.1981, Top10Acc: 0.5458, Acc: 0.0116, MacroF1: 0

### `multitask_sinusoidal_dstart_ml50_do035_s2024`


In [14]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_s2024
epoch=1, loss=2.9345
epoch=2, loss=1.7528
epoch=3, loss=1.5236
epoch=4, loss=1.4194
epoch=5, loss=1.3497
valid [task], Top5Acc: 0.3670, Top10Acc: 0.7876, Acc: 0.0591, MacroF1: 0.0826, TimeMAE: 75569.1216, TimeRMSE: 292474.2494, TimeMedAE: 3374.3318
valid [full], NDCG@5: 0.5725, HR@5: 0.7269, NDCG@10: 0.6378, HR@10: 0.9312, MRR: 0.5527
valid [sampled], NDCG@5: 0.3955, HR@5: 0.4235, NDCG@10: 0.4341, HR@10: 0.5432, MRR: 0.4179
test [task], Top5Acc: 0.3659, Top10Acc: 0.6782, Acc: 0.0891, MacroF1:

### `multitask_sinusoidal_dstart_ml50_do035_s7`


In [15]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_s7
epoch=1, loss=3.4360
epoch=2, loss=1.7939
epoch=3, loss=1.5688
epoch=4, loss=1.4338
epoch=5, loss=1.3712
valid [task], Top5Acc: 0.3716, Top10Acc: 0.6634, Acc: 0.0151, MacroF1: 0.0686, TimeMAE: 70794.8963, TimeRMSE: 291115.4016, TimeMedAE: 886.9716
valid [full], NDCG@5: 0.5348, HR@5: 0.6536, NDCG@10: 0.6347, HR@10: 0.9795, MRR: 0.5348
valid [sampled], NDCG@5: 0.4060, HR@5: 0.4222, NDCG@10: 0.4314, HR@10: 0.5015, MRR: 0.4256
test [task], Top5Acc: 0.0937, Top10Acc: 0.5623, Acc: 0.0364, MacroF1: 0.0

## Train refine sinusoidal multi-task w0.1 runs


### `multitask_sinusoidal_dstart_ml50_do035_w01_s42`


In [16]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_w01_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_w01_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_w01_s42
epoch=1, loss=0.9068
epoch=2, loss=0.4368
epoch=3, loss=0.3487
epoch=4, loss=0.3058
epoch=5, loss=0.2774
valid [task], Top5Acc: 0.5048, Top10Acc: 0.7791, Acc: 0.0623, MacroF1: 0.0958, TimeMAE: 79929.9066, TimeRMSE: 285208.0534, TimeMedAE: 1119.9765
valid [full], NDCG@5: 0.6554, HR@5: 0.7747, NDCG@10: 0.7251, HR@10: 0.9946, MRR: 0.6459
valid [sampled], NDCG@5: 0.5599, HR@5: 0.5621, NDCG@10: 0.5657, HR@10: 0.5806, MRR: 0.5747
test [task], Top5Acc: 0.2884, Top10Acc: 0.4475, Acc: 0.0137, Ma

### `multitask_sinusoidal_dstart_ml50_do035_w01_s2024`


In [17]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_w01_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_w01_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_w01_s2024
epoch=1, loss=0.8974
epoch=2, loss=0.4363
epoch=3, loss=0.3457
epoch=4, loss=0.3012
epoch=5, loss=0.2756
valid [task], Top5Acc: 0.4869, Top10Acc: 0.6866, Acc: 0.0549, MacroF1: 0.0826, TimeMAE: 80505.4291, TimeRMSE: 295100.8333, TimeMedAE: 1018.7337
valid [full], NDCG@5: 0.6336, HR@5: 0.7181, NDCG@10: 0.7148, HR@10: 0.9714, MRR: 0.6420
valid [sampled], NDCG@5: 0.5580, HR@5: 0.5632, NDCG@10: 0.5667, HR@10: 0.5903, MRR: 0.5716
test [task], Top5Acc: 0.3052, Top10Acc: 0.6540, Acc: 0.0144, 

### `multitask_sinusoidal_dstart_ml50_do035_w01_s7`


In [18]:
!python src/train_sasrec.py \
  --run_name multitask_sinusoidal_dstart_ml50_do035_w01_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_sinusoidal_multitask_w01_ndcg10_v1/multitask_sinusoidal_dstart_ml50_do035_w01_s7
epoch=1, loss=0.9279
epoch=2, loss=0.4477
epoch=3, loss=0.3558
epoch=4, loss=0.3173
epoch=5, loss=0.2914
valid [task], Top5Acc: 0.4535, Top10Acc: 0.7107, Acc: 0.0713, MacroF1: 0.1047, TimeMAE: 81441.6608, TimeRMSE: 300714.1305, TimeMedAE: 801.4105
valid [full], NDCG@5: 0.6386, HR@5: 0.7661, NDCG@10: 0.7078, HR@10: 0.9921, MRR: 0.6244
valid [sampled], NDCG@5: 0.5209, HR@5: 0.5238, NDCG@10: 0.5311, HR@10: 0.5559, MRR: 0.5385
test [task], Top5Acc: 0.2020, Top10Acc: 0.5105, Acc: 0.0392, Macr

## Load run summaries


In [19]:
import json
from pathlib import Path
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in sorted(output_path.iterdir()):
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_delta_column': config.get('time_delta_column'),
            'time_encoding': config.get('time_encoding'),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_loss_type': config.get('time_loss_type'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
            'time_sinusoidal_base': config.get('time_sinusoidal_base'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [20]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2400)
pd.set_option('display.max_colwidth', None)


## Compare refine baseline / single-task time-aware / multi-task time-aware


In [21]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
single_task_sinusoidal_runs = [
    'timeaware_dstart_sinusoidal_ml50_do035_s42',
    'timeaware_dstart_sinusoidal_ml50_do035_s2024',
    'timeaware_dstart_sinusoidal_ml50_do035_s7',
]
multitask_baseline_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]
multitask_attnbias_runs = [
    'multitask_attnbias_dstart_ml50_do035_b9_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_s7',
]
multitask_sinusoidal_runs = [
    'multitask_sinusoidal_dstart_ml50_do035_s42',
    'multitask_sinusoidal_dstart_ml50_do035_s2024',
    'multitask_sinusoidal_dstart_ml50_do035_s7',
]
multitask_sinusoidal_w01_runs = [
    'multitask_sinusoidal_dstart_ml50_do035_w01_s42',
    'multitask_sinusoidal_dstart_ml50_do035_w01_s2024',
    'multitask_sinusoidal_dstart_ml50_do035_w01_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
single_task_attnbias_df = rebuild_df(SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
single_task_sinusoidal_df = rebuild_df(SINGLE_TASK_SINUSOIDAL_NDCG10_OUTPUT_DIR)
multitask_baseline_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_attnbias_df = rebuild_df(MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR)
multitask_sinusoidal_df = rebuild_df(MULTITASK_REFINE_SINUSOIDAL_OUTPUT_DIR)
multitask_sinusoidal_w01_df = rebuild_df(MULTITASK_REFINE_SINUSOIDAL_W01_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'refine_baseline'

single_task_attnbias_subset = single_task_attnbias_df[single_task_attnbias_df['run_name'].isin(single_task_attnbias_runs)].copy()
single_task_attnbias_subset['variant'] = 'refine_attnbias_single_task'

single_task_sinusoidal_subset = single_task_sinusoidal_df[single_task_sinusoidal_df['run_name'].isin(single_task_sinusoidal_runs)].copy()
single_task_sinusoidal_subset['variant'] = 'refine_sinusoidal_single_task'

multitask_baseline_subset = multitask_baseline_df[multitask_baseline_df['run_name'].isin(multitask_baseline_runs)].copy()
multitask_baseline_subset['variant'] = 'refine_multi_task_w1.0'

multitask_attnbias_subset = multitask_attnbias_df[multitask_attnbias_df['run_name'].isin(multitask_attnbias_runs)].copy()
multitask_attnbias_subset['variant'] = 'refine_attnbias_multi_task_w1.0'

multitask_sinusoidal_subset = multitask_sinusoidal_df[multitask_sinusoidal_df['run_name'].isin(multitask_sinusoidal_runs)].copy()
multitask_sinusoidal_subset['variant'] = 'refine_sinusoidal_multi_task_w1.0'

multitask_sinusoidal_w01_subset = multitask_sinusoidal_w01_df[multitask_sinusoidal_w01_df['run_name'].isin(multitask_sinusoidal_w01_runs)].copy()
multitask_sinusoidal_w01_subset['variant'] = 'refine_sinusoidal_multi_task_w0.1'

df_compare = pd.concat(
    [
        baseline_subset,
        single_task_attnbias_subset,
        single_task_sinusoidal_subset,
        multitask_baseline_subset,
        multitask_attnbias_subset,
        multitask_sinusoidal_subset,
        multitask_sinusoidal_w01_subset,
    ],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

display_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric', 'best_epoch',
    'time_encoding', 'time_delta_column', 'time_loss_weight', 'time_prediction_target',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10', 'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy', 'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae', 'best_valid_task_time_rmse', 'best_valid_task_time_median_ae',
    'best_test_at_best_valid_task_accuracy', 'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy', 'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae', 'best_test_at_best_valid_task_time_rmse', 'best_test_at_best_valid_task_time_median_ae',
]
existing_display_cols = [c for c in display_cols if c in df_compare.columns]
df_compare[existing_display_cols]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,best_epoch,time_encoding,time_delta_column,time_loss_weight,time_prediction_target,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_mrr,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_mrr,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_top5_accuracy,best_valid_task_top10_accuracy,best_valid_task_time_mae,best_valid_task_time_rmse,best_valid_task_time_median_ae,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_top5_accuracy,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_time_median_ae
0,multitask_attnbias_dstart_ml50_do035_b9_s7,7,refine_attnbias_multi_task_w1.0,50,0.35,full_valid_ndcg@10,25,raw,delta_start_seconds,1.0,delta_next_seconds,0.701169,0.841042,0.741140,0.968127,0.674917,0.724066,0.978226,0.729992,0.995266,0.640692,0.584575,0.587549,0.589915,0.604232,0.599089,0.161131,0.165134,0.197971,0.283338,0.214905,0.064153,0.054342,0.394005,0.606402,69849.790407,280340.108895,2552.965280,0.027996,0.022689,0.382878,0.617663,12280.778068,76506.943274,301.367851
1,multitask_attnbias_dstart_ml50_do035_b9_s42,42,refine_attnbias_multi_task_w1.0,50,0.35,full_valid_ndcg@10,35,raw,delta_start_seconds,1.0,delta_next_seconds,0.705553,0.836408,0.748445,0.968937,0.684121,0.779972,0.975749,0.788496,1.000000,0.719152,0.597182,0.599023,0.602575,0.615979,0.611855,0.210614,0.246850,0.264049,0.413901,0.249278,0.061856,0.079921,0.456593,0.641888,70299.450571,281214.731226,1427.825244,0.023574,0.019707,0.318521,0.667254,11338.988327,72120.057406,75.289802
2,multitask_attnbias_dstart_ml50_do035_b9_s2024,2024,refine_attnbias_multi_task_w1.0,50,0.35,full_valid_ndcg@10,30,raw,delta_start_seconds,1.0,delta_next_seconds,0.688210,0.807562,0.739497,0.969508,0.672990,0.774766,0.953504,0.790523,0.999864,0.721091,0.584314,0.588291,0.592206,0.612956,0.598530,0.223472,0.271520,0.297880,0.502508,0.259729,0.059900,0.039673,0.451281,0.707955,78572.105940,291972.478347,10619.751266,0.117256,0.046148,0.608920,0.633455,15157.516227,76170.374705,108.494928
3,attnbias_dstart_ml50_do035_b9_s7,7,refine_attnbias_single_task,50,0.35,full_valid_ndcg@10,40,raw,delta_start_seconds,None,None,0.714870,0.824671,0.755366,0.953861,0.699184,0.846665,0.999728,0.846762,1.000000,0.793978,0.607399,0.615479,0.623265,0.666077,0.623619,0.442098,0.450204,0.497216,0.628223,0.485290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,attnbias_dstart_ml50_do035_b9_s42,42,refine_attnbias_single_task,50,0.35,full_valid_ndcg@10,20,raw,delta_start_seconds,None,None,0.709565,0.864487,0.741856,0.969750,0.673783,0.764719,0.935724,0.785724,0.999865,0.714860,0.566501,0.577970,0.585005,0.635957,0.585189,0.115783,0.193920,0.199334,0.453522,0.156593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,attnbias_dstart_ml50_do035_b9_s2024,2024,refine_attnbias_single_task,50,0.35,full_valid_ndcg@10,20,raw,delta_start_seconds,None,None,0.702964,0.865254,0.735385,0.971475,0.665031,0.863911,0.928234,0.885738,0.999865,0.850981,0.548977,0.558247,0.566216,0.612548,0.568592,0.366083,0.436179,0.428086,0.626287,0.388188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,refine_ml50_do035_s7,7,refine_baseline,50,0.35,full_valid_ndcg@10,25,None,delta_prev_seconds,None,None,0.696223,0.857259,0.728826,0.961737,0.660451,0.859037,0.957678,0.872416,1.000000,0.831208,0.565802,0.575315,0.57

In [22]:
summary_metric_cols = [
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10', 'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy', 'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae', 'best_valid_task_time_rmse', 'best_valid_task_time_median_ae',
    'best_test_at_best_valid_task_accuracy', 'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy', 'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae', 'best_test_at_best_valid_task_time_rmse', 'best_test_at_best_valid_task_time_median_ae',
]
summary_metric_cols = [c for c in summary_metric_cols if c in df_compare.columns]
summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_mrr           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_mrr           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_mrr           best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_top5_accuracy           best_valid_task_top10_accuracy           best_valid_task_time_mae               best_valid_task_time_rmse              best_valid_task_time_median_ae              best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_top5_accuracy           best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_time_median_ae            
                                                    mean       std                 mean       std                    mean       std                  mean       std                mean       std                                mean       std                              mean       std                                 mean       std                               mean       std                             mean       std                      mean       std                    mean       std                       mean       std                     mean       std                   mean       std                                   mean       std                                 mean       std                                    mean       std                                  mean       std                                mean       std                     mean       std                     mean       std                          mean       std                           mean       std                     mean           std                      mean          std                           mean          std                                  mean       std                                  mean       std                                       mean       std                                        mean       std                                  mean          std                                   mean          std                                        mean         std
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## What to look at

Read the results in this order:
- compare `refine_sinusoidal_single_task` vs `refine_sinusoidal_multi_task_w1.0`
- compare `refine_sinusoidal_multi_task_w1.0` vs `refine_sinusoidal_multi_task_w0.1`
- compare `refine_sinusoidal_multi_task_w1.0` vs `refine_attnbias_multi_task_w1.0`
- use `best_test_at_best_valid_full_ndcg@10` as the main ranking metric
- use `best_test_at_best_valid_task_time_mae` as the main next-time metric
